# Day 4 · Matplotlib / Seaborn 数据可视化

目标：用图回答具体问题，每张图写出“观察、证据、局限”。
基础 50 分钟、深度 30 分钟、广度 25 分钟、作业 35 分钟；Bonus 30 分钟。
所有探索仅使用训练集。图中文字使用英文，解释使用中文，避免依赖中文字体。

### 1. Matplotlib 的 Figure 与 Axes

Figure 是整张画布，Axes 是一个绘图区。plt.subplots 返回它们。
Seaborn 在 Matplotlib 上提供按字段作图的接口。先明确问题，再选择图形：
分布用直方图，分组比较用箱线图，两个数值的关系用散点图。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "course/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(data=train, x="G3", bins=np.arange(-0.5,21.5,1), ax=ax)
ax.set(title="Final grade distribution", xlabel="Grade (0-20)", ylabel="Students")
fig.tight_layout()
plt.show()

练习：把 bins 改为 5 和 20，描述读图感受的变化。样本相同，但分箱影响视觉形状。
注意零分是真实记录，图上出现零分不能直接作为删除理由。

### 2. 四张图构成一次探索

箱线图显示中位数、四分位数和离群标记，不能自动判断录入错误。
散点的重叠可以通过 alpha 减轻。
相关系数描述线性关系；接近零也可能存在非线性关系。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=train, x="G3", bins=np.arange(-0.5,21.5,1), ax=axes[0,0])
axes[0,0].set_title("Training final grades (0-20)")
sns.boxplot(data=train, x="studytime", y="G3", ax=axes[0,1])
axes[0,1].set_title("Study-time categories vs grade")
sns.scatterplot(data=train, x="G2", y="G3", alpha=0.5, ax=axes[1,0])
axes[1,0].set_title("Second-period vs final grade")
sns.heatmap(train[["G1","G2","G3","studytime","failures"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1,1])
axes[1,1].set_title("Pearson correlation (training only)")
fig.tight_layout()
plt.show()


In [ ]:
display(train.groupby("studytime")["G3"].agg(["count","mean","median"]))
display(train[["G1","G2","G3","studytime","failures"]].corr()["G3"].sort_values())

### 3. 深度：相关不等于因果

学习时间与成绩的关联可能同时受已有基础、家庭背景等因素影响。
不能根据分组均值断言“多学习一档会提高多少分”。
G2 与 G3 的相关性很强也不等于开学时就可以使用 G2。
解释模板：在训练样本中，___；证据是图/统计量___；但___限制了推论。

散点中的残差、非线性和不同范围的波动，比“看起来有趋势”更值得记录。

In [ ]:
# 人工反例：相关系数接近零，但存在确定的非线性关系。
x = np.linspace(-3,3,101)
y = x**2
print("Pearson r:", np.corrcoef(x,y)[0,1])
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(x,y,s=15)
ax.set(xlabel="x", ylabel="x squared", title="Nonlinear relation with near-zero correlation")
plt.show()

### 4. 广度：分组均值与图表导出

柱图从零起步通常更容易比较数量。均值表必须带样本数。
不要用连接无序类别的折线暗示连续变化。
保存图像要在关闭图之前，用 fig.savefig；生成文件进入 outputs，不进入原始数据目录。

In [ ]:
stats = train.groupby("school")["G3"].agg(["count","mean"])
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(stats.index, stats["mean"])
ax.set(ylim=(0,20), ylabel="Mean grade", title="Mean grade by school")
for i, (school, row) in enumerate(stats.iterrows()):
    ax.text(i, row["mean"] + 0.4, f"n={int(row['count'])}", ha="center")
fig.tight_layout()
output = ROOT / "course/bootcamp/week1/outputs"
output.mkdir(exist_ok=True)
fig.savefig(output / "day4_school_means.png", dpi=150, bbox_inches="tight")
plt.show()

### DAY 4 ASSIGNMENT

基础：画四种图，每图补一句观察。

深度：解释 G3=0 为什么保留；说明一个不能由图表推断的因果结论。

广度：对比 studytime 分组的 count 和 mean，解释小样本组的局限。

Bonus：绘制 G2/G3 的联合分布；用 hue="school" 检查整体相关性是否掩盖组间差异。

验收：图标题和单位明确；不读取测试集来选择特征；保存一张 PNG。

In [ ]:
# 作业区：请写你自己的四条观察。


### 参考答案示例

直方图可看集中区间及零分记录；散点图可看总体趋势及偏离趋势的个例。
具体数值需以本 Notebook 输出为准，不预设学习时间效应一定为正。

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
sns.scatterplot(data=train, x="G2", y="G3", hue="school", alpha=0.6, ax=ax)
ax.set(title="Grade relationship by school")
plt.show()
print("训练集零分人数：", int((train["G3"] == 0).sum()))

### 复盘（填写）

我最有证据的三条发现：___。还需哪些信息才能验证因果关系：___。